# OpenPlaque UCLA TPV with Tuned Default Boundary Refinement

Runs from a clean Colab environment.

This notebook uses tuned boundary-refinement defaults:

```python
DEFAULT_REFINEMENT_PARAMS = {'remove_small': True, 'min_component_voxels': 80, 'trim_lumen_adjacent': False, 'lumen_distance_voxels': 0, 'erode_core': False, 'erosion_iterations': 0, 'low_hu_threshold': None, 'high_hu_threshold': None}
```

Research use only. Not for clinical decision-making.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
!git clone https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque || true
!git -C /content/OpenPlaque pull


In [ ]:
!wget -q https://raw.githubusercontent.com/pazzani/OpenPlaque/main/requirements-colab.txt -O /content/requirements-colab.txt
!pip install -q -r /content/requirements-colab.txt


In [ ]:
import os, sys
from pathlib import Path

repo = Path("/content/OpenPlaque")
src = repo / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

os.environ["nnUNet_raw"] = "/content/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/content/nnUNet_preprocessed"
os.environ["nnUNet_results"] = "/content/nnUNet_results"
for d in [os.environ["nnUNet_raw"], os.environ["nnUNet_preprocessed"], os.environ["nnUNet_results"]]:
    os.makedirs(d, exist_ok=True)

print("OpenPlaque path:", src)


In [ ]:
!nvidia-smi


In [ ]:
import zipfile
from pathlib import Path

model_zip = Path("/content/drive/MyDrive/OpenPlaque/models/Dataset001_CCTA_DHM-20260703T233210Z-3-001.zip")
model_target = Path("/content/nnUNet_results/Dataset001_CCTA_DHM")

if model_target.exists():
    print("Model already extracted:", model_target)
else:
    if not model_zip.exists():
        raise FileNotFoundError(f"Missing model ZIP: {model_zip}")
    with zipfile.ZipFile(model_zip) as z:
        z.extractall("/content/nnUNet_results")
    print("Extracted model")


In [ ]:
from openplaque.study import OpenPlaqueStudy
from openplaque.segmentation import segment_vessel
from openplaque.boundary import refine_plaque_mask, DEFAULT_REFINEMENT_PARAMS

print("Default refinement parameters:")
print(DEFAULT_REFINEMENT_PARAMS)

study = OpenPlaqueStudy("/content/drive/MyDrive/OpenPlaque/Full_DICOM.zip")
study.summary()


## Load curved coronary reformats

In [ ]:
image_rca, volume_rca, _ = study.load_series(1035)
image_lcx, volume_lcx, _ = study.load_series(1039)
image_lad, volume_lad, _ = study.load_series(1043)

print("RCA:", volume_rca.shape, image_rca.GetSpacing())
print("LCX:", volume_lcx.shape, image_lcx.GetSpacing())
print("LAD:", volume_lad.shape, image_lad.GetSpacing())


## Run segmentation

In [ ]:
lad_report = segment_vessel(image_lad, volume_lad, "LAD")
rca_report = segment_vessel(image_rca, volume_rca, "RCA")
lcx_report = segment_vessel(image_lcx, volume_lcx, "LCX")

for r in [lad_report, rca_report, lcx_report]:
    r.summary()
    r.show_overlay(label=2)


## Apply tuned default boundary refinement

In [ ]:
reports = [lad_report, rca_report, lcx_report]

refinements = {}
for r in reports:
    refinements[r.name] = refine_plaque_mask(
        volume=r.volume,
        mask=r.mask,
        spacing=r.mask_image.GetSpacing(),
        use_defaults=True,
    )
    print("\n" + "=" * 40)
    print(r.name)
    refinements[r.name].summary()
    refinements[r.name].show_refined_overlay(r.volume)


## Raw vs refined total plaque volume

In [ ]:
raw_total = sum(r.tpv_mm3 for r in reports)
refined_total = sum(refinements[r.name].refined_tpv_mm3 for r in reports)
removed_total = sum(refinements[r.name].removed_volume_mm3 for r in reports)

print("Raw vs refined plaque volume")
print("-" * 70)
print(f"{'Vessel':<6} {'Raw TPV':>12} {'Refined TPV':>14} {'Removed':>12}")
print("-" * 70)
for r in reports:
    ref = refinements[r.name]
    print(f"{r.name:<6} {r.tpv_mm3:12.2f} {ref.refined_tpv_mm3:14.2f} {ref.removed_volume_mm3:12.2f}")
print("-" * 70)
print(f"{'TOTAL':<6} {raw_total:12.2f} {refined_total:14.2f} {removed_total:12.2f}")


## Save outputs

In [ ]:
import os

save_dir = "/content/drive/MyDrive/OpenPlaque/Segmentations"
os.makedirs(save_dir, exist_ok=True)

for r in reports:
    raw_path = f"{save_dir}/{r.name}_raw_plaque_segmentation.nii.gz"
    ref_path = f"{save_dir}/{r.name}_tuned_refined_plaque_segmentation.nii.gz"
    r.save_mask(raw_path)
    refinements[r.name].save_refined_mask(r.mask_image, ref_path)

summary_path = f"{save_dir}/tpv_tuned_refinement_summary.txt"
with open(summary_path, "w") as f:
    f.write("OpenPlaque UCLA TPV with tuned default boundary refinement\n")
    f.write("Research use only. Not for clinical decision-making.\n\n")
    f.write(f"Default params: {DEFAULT_REFINEMENT_PARAMS}\n\n")
    for r in reports:
        ref = refinements[r.name]
        f.write(f"{r.name}: raw={r.tpv_mm3:.2f} mm3, refined={ref.refined_tpv_mm3:.2f} mm3, removed={ref.removed_volume_mm3:.2f} mm3\n")
    f.write(f"RAW TOTAL TPV: {raw_total:.2f} mm3\n")
    f.write(f"REFINED TOTAL TPV: {refined_total:.2f} mm3\n")
    f.write(f"REMOVED VOLUME: {removed_total:.2f} mm3\n")

print("Saved:", summary_path)
